In [ ]:
# import required libraries
!pip install wget
import os
import os.path
import numpy as np
import pandas as pd
from math import sqrt
from heapq import nlargest
from tqdm import trange
from tqdm import tqdm
from scipy import stats
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import wget

from scipy.sparse import coo_matrix
from scipy.sparse.linalg import svds

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=99c8d9f380653d03dc21a3f9c45bd29385cb9b5cdaa93e2ddb191f31d3016a4e
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [ ]:
wget.download("https://github.com/MIE451-1513-2019/course-datasets/raw/master/ml-100k.zip")
!unzip ml-100k.zip
MOVIELENS_DIR = "ml-100k"

Archive:  ml-100k.zip
   creating: ml-100k/
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.base         
  inflating: ml-100k/u2.test         
  inflating: ml-100k/u3.base         
  inflating: ml-100k/u3.test         
  inflating: ml-100k/u4.base         
  inflating: ml-100k/u4.test         
  inflating: ml-100k/u5.base         
  inflating: ml-100k/u5.test         
  inflating: ml-100k/ua.base         
  inflating: ml-100k/ua.test         
  inflating: ml-100k/ub.base         
  inflating: ml-100k/ub.test         


In [ ]:
!ls {MOVIELENS_DIR}

allbut.pl  u1.base  u2.test  u4.base  u5.test  ub.base	u.genre  u.occupation
mku.sh	   u1.test  u3.base  u4.test  ua.base  ub.test	u.info	 u.user
README	   u2.base  u3.test  u5.base  ua.test  u.data	u.item


In [ ]:
def getData(folder_path, file_name):
    fields = ['userID', 'itemID', 'rating', 'timestamp']
    data = pd.read_csv(os.path.join(folder_path, file_name), sep='\t', names=fields)
    return data

In [ ]:
rating_df = getData(MOVIELENS_DIR, 'u.data')
rating_df_train = getData(MOVIELENS_DIR, 'u1.base')
rating_df_test = getData(MOVIELENS_DIR, 'u1.test')

In [ ]:
class CrossValidation(object):
    def __init__(self, metric, data_path=MOVIELENS_DIR):
        """
            INPUT:
                metric: string. from['RMSE','P@K','R@K']
        """
        self.folds = self._getData(MOVIELENS_DIR)
        self.metric_name = metric
        self.metric = self._getMetric(self.metric_name)

    def _getMetric(self, metric_name):
        """
            Don't change this
        """
        switcher = {
            'RMSE': self.rmse,
            'P@K': self.patk,
            'R@K': self.ratk,
            'RPrecision': self.rprecision
        }

        return switcher[metric_name]

    @staticmethod
    def rmse(data, k, num_users, num_items, pred, true='rating'):
        """
            data: pandas DataFrame.
            pred: string. Column name that corresponding to the prediction
            true: string. Column name that corresponding to the true rating
        """
        return sqrt(mean_squared_error(data[pred], data[true]))

    # Precision at k
    def patk(self, data, k, num_users, num_items, pred, true='rating'):
        """
            data: pandas DataFrame.
            k: top-k items retrived
            pred: string. Column name that corresponding to the prediction
            true: string. Column name that corresponding to the true rating
        """
        prediction = self.getMatrix(data, num_users, num_items, pred)
        testSet =  self.getMatrix(data, num_users, num_items, true)

        # Initialize sum and count vars for average calculation
        sumPrecisions = 0
        countPrecisions = 0

        # Define function for converting 1-5 rating to 0/1 (like / don't like)
        vf = np.vectorize(lambda x: 1 if x >= 4 else 0)

        for userID in range(num_users):
            # Pick top K based on predicted rating
            userVector = prediction[userID,:]
            topK = nlargest(k, range(len(userVector)), userVector.take)

            # Convert test set ratings to like / don't like
            userTestVector = vf(testSet[userID,:]).nonzero()[0]

            # Calculate precision
            precision = float(len([item for item in topK if item in userTestVector]))/len(topK)

            # Update sum and count
            sumPrecisions += precision
            countPrecisions += 1

        # Return average P@k
        return float(sumPrecisions)/countPrecisions

    # Recall at k
    def ratk(self, data, k, num_users, num_items, pred, true='rating'):
        """
            data: pandas DataFrame.
            k: top-k items relevant
            pred: string. Column name that corresponding to the prediction
            true: string. Column name that corresponding to the true rating
        """
        prediction = self.getMatrix(data, num_users, num_items, pred)
        testSet =  self.getMatrix(data, num_users, num_items, true)
        # Initialize sum and count vars for average calculation
        sumRecalls = 0
        countRecalls = 0

        # Define function for converting 1-5 rating to 0/1 (like / don't like)
        vf = np.vectorize(lambda x: 1 if x >= 4 else 0)

        for userID in range(num_users):
            # Pick top K based on predicted rating
            userVector = prediction[userID,:]
            topK = nlargest(k, range(len(userVector)), userVector.take)

            # Convert test set ratings to like / don't like
            userTestVector = vf(testSet[userID,:]).nonzero()[0]

            # Ignore user if has no ratings in the test set
            if (len(userTestVector) == 0):
                continue

            # Calculate recall
            recall = float(len([item for item in topK if item in userTestVector]))/len(userTestVector)

            # Update sum and count
            sumRecalls += recall
            countRecalls += 1

        # Return average R@k
        return float(sumRecalls)/countRecalls

    def rprecision(self, data, k, num_users, num_items, pred, true='rating'):
        """
            data: pandas DataFrame.
            k: top-k items relevant
            pred: string. Column name that corresponding to the prediction
            true: string. Column name that corresponding to the true rating
        """
        prediction = self.getMatrix(data, num_users, num_items, pred)
        testSet = self.getMatrix(data, num_users, num_items, true)
        # Initialize sum and count vars for average calculation
        sumRPs = 0
        countRPs = 0

        # Define function for converting 1-5 rating to 0/1 (like / don't like)
        vf = np.vectorize(lambda x: 1 if x >= 4 else 0)

        for userID in range(num_users):
            # Pick top K based on predicted rating
            userVector = prediction[userID, :]


            # Convert test set ratings to like / don't like
            userTestVector = vf(testSet[userID, :]).nonzero()[0]

            # Ignore user if has no ratings in the test set
            if (len(userTestVector) == 0):
                continue

            topK = nlargest(len(userTestVector), range(len(userVector)), userVector.take)
            # Calculate recall
            rp = float(len([item for item in topK if item in userTestVector])) / len(userTestVector)

            # Update sum and count
            sumRPs += rp
            countRPs += 1

        # Return average R@k
        return float(sumRPs) / countRPs

    @staticmethod
    def getMatrix(rating_df, num_users, num_items, column_name):
        matrix = np.zeros((num_users, num_items))

        for (index, userID, itemID, value) in rating_df[['userID','itemID', column_name]].itertuples():
            matrix[userID-1, itemID-1] = value

        return matrix

    @staticmethod
    def _getData(data_path):
        """
            Don't change this function
        """
        folds = []
        data_types = ['u{0}.base','u{0}.test']
        for i in range(1,6):
            train_set = getData(data_path, data_types[0].format(i))
            test_set = getData(data_path, data_types[1].format(i))
            folds.append([train_set, test_set])
        return folds

    def run(self, algorithms, num_users, num_items, k=1):
        """
            5-fold cross-validation
            algorithms: list. a list of algorithms.
                        eg: [user_cosine_recsys, item_euclidean_recsys]
        """

        scores = {}
        for algorithm in algorithms:
            print('Processing algorithm {0}'.format(algorithm.getPredColName()))
            fold_scores = []
            for fold in self.folds:
                algorithm.reset()
                algorithm.predict_all(fold[0], num_users, num_items)
                prediction = algorithm.evaluate_test(fold[1])
                pred_col = algorithm.getPredColName()
                fold_scores.append(self.metric(prediction, k, num_users, num_items, pred_col))

            mean = np.mean(fold_scores)
            ci_low, ci_high = stats.t.interval(0.95, len(fold_scores)-1, loc=mean, scale=stats.sem(fold_scores))
            scores[algorithm.getPredColName()] = [fold_scores, mean, ci_low, ci_high]

        results = scores

        return results


## Q9

In [ ]:
def dataPreprocessor(rating_df, num_users, num_items):
    """
        INPUT:
            data: pandas DataFrame. columns=['userID', 'itemID', 'rating' ...]
            num_row: int. number of users
            num_col: int. number of items

        OUTPUT:
            matrix: 2D numpy array.

        NOTE 1: see where something very similar is done in the lab in function 'buildUserItemMatrix'

        NOTE 2: data can have more columns, but your function should ignore
              additional columns.
    """
    ########### your code goes here ###########
    matrix = np.zeros((num_users, num_items), dtype=np.int8)

    # Populate the matrix based on the dataset
    for (index, userID, itemID, rating, timestamp) in rating_df.itertuples():
        matrix[userID-1, itemID-1] = rating

    ###########         end         ###########
    return matrix

In [ ]:
class CompetitionRecSys(object):
    """
    You can define new methods if you need. Don't use global variables in the class.
    """
    def __init__(self, num_feat=35, epsilon=1.75, _lambda=0.18, momentum=0.75, maxepoch=50, num_batches=160, batch_size=1700 ):
    #def __init__(self, num_feat=35, epsilon=1.75, _lambda=0.175, momentum=0.75, maxepoch=100, num_batches=160, batch_size=1700 ):
        """
        Initialization of the class
        1. Make sure to fill out self.pred_column_name, the name you give  to your competition method

        """
        ########## your code goes here ###########

        self.rmse_train = []
        self.rmse_test = []
        self.pred_column_name='PMF'

        self.num_feat = num_feat
        self.epsilon = epsilon
        self._lambda = _lambda
        self.momentum = momentum
        self.maxepoch = maxepoch
        self.num_batches = num_batches
        self.batch_size = batch_size
        self.test = False

        self.pred_column_name = 'PMF'
        self.w_Item = None
        self.w_User = None
        self.b_User = None
        self.b_Item = None

        self.rmse_train = []
        self.rmse_test = []

        ###########         end         ###########

    def predict_all(self, train_vec, num_user, num_item):

        """
        INPUT:
            data: pandas DataFrame. columns=['userID', 'itemID', 'rating'...]
            num_user: scalar. number of users
            num_item: scalar. number of items
        OUTPUT:
            no return...

        NOTES:
            This function is where you train your model
        """

        ########## your code goes here ###########
        train_vec = train_vec.iloc[:, :3].values

        if self.test:

            train_vec, val_vec = train_test_split(train_vec, test_size=0.1)
            pairs_val = val_vec.shape[0]
            self.mean_rating_test = np.mean(val_vec[:, 2])

        self.mean_rating_train = np.mean(train_vec[:, 2])
        pairs_train = train_vec.shape[0]

        # sparse rating matrix
        rows = train_vec[:, 0].astype(int) - 1
        cols = train_vec[:, 1].astype(int) - 1
        vals = train_vec[:, 2]
        R = coo_matrix((vals, (rows, cols)), shape=(num_user, num_item)).toarray()

        # using svd for warm start
        U, sigma, Vt = svds(R, k=self.num_feat)
        sigma = np.diag(sigma)
        self.w_User = np.dot(U, np.sqrt(sigma))
        self.w_Item = np.dot(Vt.T, np.sqrt(sigma))

        # bias term added
        self.b_User = np.zeros(num_user)
        self.b_Item = np.zeros(num_item)
        self.global_mean = self.mean_rating_train

        # momentum
        self.w_Item_inc = np.zeros((num_item, self.num_feat))
        self.w_User_inc = np.zeros((num_user, self.num_feat))
        self.b_User_inc = np.zeros(num_user)
        self.b_Item_inc = np.zeros(num_item)

        # train
        for epoch in range(self.maxepoch):
            shuffled_order = np.arange(pairs_train)
            np.random.shuffle(shuffled_order)

            for batch in range(self.num_batches):
                idx = np.arange(self.batch_size * batch, self.batch_size * (batch + 1))
                batch_idx = np.mod(idx, shuffled_order.shape[0])


                u_id = train_vec[shuffled_order[batch_idx], 0].astype(int) - 1
                i_id = train_vec[shuffled_order[batch_idx], 1].astype(int) - 1

                r_ui = train_vec[shuffled_order[batch_idx], 2]

                # preidct
                pred = np.sum(self.w_User[u_id] * self.w_Item[i_id], axis=1) \
                        + self.global_mean + self.b_User[u_id] + self.b_Item[i_id]
                err = pred - r_ui

                # gradient
                m_1 = 2 * (err[:, np.newaxis] * self.w_Item[i_id]) + self._lambda * self.w_User[u_id]
                m_2 = 2 * (err[:, np.newaxis] * self.w_User[u_id]) + self._lambda * self.w_Item[i_id]
                mb_1 = 2 * err + self._lambda * self.b_User[u_id]
                mb_2 = 2 * err + self._lambda * self.b_Item[i_id]

                # aggregate grad
                dw_1 = np.zeros_like(self.w_User)
                dw_2 = np.zeros_like(self.w_Item)
                db_1 = np.zeros_like(self.b_User)
                db_2 = np.zeros_like(self.b_Item)

                for j in range(len(u_id)):
                    dw_1[u_id[j]] +=m_1[j]
                    dw_2[i_id[j]] +=m_2[j]
                    db_1[u_id[j]] +=mb_1[j]
                    db_2[i_id[j]] +=mb_2[j]

                # update w momentum
                self.w_User_inc = self.momentum * self.w_User_inc+self.epsilon * dw_1/self.batch_size
                self.w_Item_inc = self.momentum * self.w_Item_inc+self.epsilon * dw_2/self.batch_size
                self.b_User_inc = self.momentum * self.b_User_inc+self.epsilon * db_1/self.batch_size
                self.b_Item_inc = self.momentum * self.b_Item_inc+self.epsilon * db_2/self.batch_size

                self.w_User -= self.w_User_inc
                self.w_Item -= self.w_Item_inc
                self.b_User -= self.b_User_inc
                self.b_Item -= self.b_Item_inc

            # rmse
            u_all = train_vec[:, 0].astype(int) - 1
            i_all = train_vec[:, 1].astype(int) - 1
            pred_all = np.sum(self.w_User[u_all] * self.w_Item[i_all], axis=1) \
                       + self.global_mean + self.b_User[u_all] + self.b_Item[i_all]
            rmse = np.sqrt(np.mean((pred_all - train_vec[:, 2])**2))
            self.rmse_train.append(rmse)



    def evaluate_test(self, test_df, copy=False):
        """
            INPUT:
                data: pandas DataFrame. columns=['userID', 'itemID', 'rating'...]
            OUTPUT:
                predictions:  pandas DataFrame.
                              columns=['userID', 'itemID', 'rating', 'base-method'...]

            NOTES:
            This function is where your model makes prediction
            Please fill out: prediction.loc[index, self.pred_column_name] = None

        """
        if copy:
            prediction = pd.DataFrame(test_df.copy(), columns=['userID', 'itemID', 'rating'])
        else:
            prediction = pd.DataFrame(test_df, columns=['userID', 'itemID', 'rating'])
        prediction[self.pred_column_name] = np.nan

        for (index,
             userID,
             itemID) in tqdm(prediction[['userID','itemID']].itertuples()):
            ########### your code goes here ###########
            #prediction.loc[index, self.pred_column_name] = None
            u = int(userID) -1
            i = int(itemID) - 1
            pred = (np.dot(self.w_User[u], self.w_Item[i]) + self.global_mean + self.b_User[u] + self.b_Item[i])
            prediction.loc[index, self.pred_column_name] = pred
            # prediction.loc[index, self.pred_column_name] = (np.dot(self.w_Item, self.w_User[int(userID), :]) + self.mean_rating_train)[int(itemID)]
            ###########         end         ###########

        return prediction


    def getPredColName(self):
        """
            return prediction column name
        """
        return self.pred_column_name

    def reset(self):
        """
            reuse the instance of the class by removing model
        """
        ########### your code goes here ###########
        try:
            self.w_Item = None
            self.w_User = None
            self.b_Item = None
            self.b_User = None
        except:
            print("You do not have w_Item, w_User")

        ##########         end         ###########



In [ ]:
competition = CompetitionRecSys()
algorithm_instances = [competition]
cv_rp = CrossValidation('RPrecision')
rp = cv_rp.run(algorithm_instances,  len(rating_df.userID.unique()), len(rating_df.itemID.unique()))

Processing algorithm PMF


20000it [00:02, 7652.60it/s]
20000it [00:02, 7666.79it/s]
20000it [00:02, 7005.54it/s]
20000it [00:02, 7316.67it/s]
20000it [00:02, 7371.59it/s]


In [ ]:
rp

{'PMF': [[0.7210067212402419,
   0.7123285056059364,
   0.7263543114609448,
   0.7303697775891285,
   0.7480891278225611],
  np.float64(0.7276296887437625),
  np.float64(0.7111371230906929),
  np.float64(0.7441222543968322)]}

# Validation

In [ ]:
# Constants for validation only
ROW_NUM = 943
COL_NUM = 1682
RATING_COL = 'rating'

In [ ]:
def validateDataPreprocessor(path=MOVIELENS_DIR, getData=getData, getMatrix=CrossValidation.getMatrix):
    validation_df = getData(MOVIELENS_DIR, 'u1.test')
    try:
        matrix = getMatrix(validation_df, ROW_NUM, COL_NUM, RATING_COL)
    except:
        print('dataPreprocessor function has error')
        return
    try:
        assert(matrix.shape == (ROW_NUM,COL_NUM)),\
        "Shape of matrix{0} doesn't match predefined shape (943,1682)".format(matrix.shape)
    except Exception as e:
        print(e)
    return validation_df

In [ ]:
validation_df = validateDataPreprocessor()

**Approach explained**

My approach borrows from Q4 of the assignment and improves the pmf application.
A major difference is that instead of gaussian initialisation, I use a warm start using svd so the user and item latent vectors are initialized from a truncated SVD of the sparse rating matrix. momentum, mini batches and l2 regularisation is the same as pmf from q4. I also added user and item biases to capture patterns in ratings (e.g., some users rate higher on average, some items are consistently rated higher).

I also used grid search to find parameters that would improve performance.